# 属性条件付き予測と公平性：HAM10000 の ERM・GroupDRO

CheXpert の [requested_figures.ipynb](requested_figures.ipynb) と同じ 6 種類の図を、HAM10000 で作る。
群は sex × 年齢 65 歳の 4 群、seed 42/43/44。ResNet-50 / Spatial LoRA (stage3–4 + fc) / Spatial LoRA (fc only) を比較する。
図1–4は **best validation AUROC checkpoint の test**、図6は **validation の学習推移**。

CheXpert との違い：

- 陽性は**悪性**（mel / akiec）。害の大きい側は見落としなので、最悪群は **TPR が最小の群**（worst-TPR）、群間差は **Eopp1（TPR の差）** を主に見る。
- test の 294 件は sex か age を欠き群を決められない。全体指標も含めて**属性の揃った 1,217 件**で計算する（全行の AUROC は CSV に別に残す）。
- **EO rate は回し直し中なので含めない。** 揃ったら `OBJECTIVES` に足して再実行する。

エラーバー／帯は 3 seed の標本 SD で、標本誤差や有意差を表さない。test の陽性は群ごとに 34〜62 件しかない。

再実行：`uv run jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=hypernet-fairness analysis/hypernet_conditioning_grid/requested_figures_ham10000.ipynb`。
予測 cache が無ければ `analysis/common/predictions.py --study hypernet_conditioning_grid --split {test,val}` で作る。

In [ ]:
import base64
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from matplotlib.backends.backend_pdf import PdfPages

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.hypernet_conditioning_grid.requested_analysis_ham10000 import (  # noqa: E402
    GROUP_LABELS,
    KEYS,
    learning_tables,
    load_split,
    prediction_tables,
    runs,
)

PACKAGE = study_dir("hypernet_conditioning_grid")
RESULTS = PACKAGE / "results" / "requested_figures_ham10000"
FIGURES = PACKAGE / "figures" / "requested_figures_ham10000"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
plt.style.use(STYLE_SHEET)
BOOK = PdfPages(FIGURES / "all_requested_figures.pdf")
MODEL = {"resnet": "ResNet-50", "spatial_lora": "Spatial LoRA (S3–4 + fc)", "spatial_lora_fc": "Spatial LoRA (fc only)"}
STAGE = {"one_stage": "1-stage", "two_stage": "2-stage (frozen ResNet)"}
OBJECTIVE = {"erm": "ERM", "group_dro": "GroupDRO"}
COLORS = {"erm": "#0173B2", "group_dro": "#DE8F05", "eo_rate": "#029E73"}
FAIRNESS = ["auroc_gap", "eopp1", "worst_group_auroc", "min_tpr"]
GROUP_METRICS = ["auroc", "bacc", "tpr", "fpr"]
LOWER_IS_BETTER = ("auroc_gap", "eopp1", "fpr")
TITLES = {
    "auroc_gap": "AUC gap ↓",
    "eopp1": "Eopp1 (TPR gap) ↓",
    "worst_group_auroc": "Worst-AUC ↑",
    "min_tpr": "Worst-TPR ↑",
    "auroc": "AUROC ↑",
    "bacc": "BACC ↑",
    "tpr": "TPR ↑",
    "fpr": "FPR ↓",
}
MODELS = list(MODEL)
OBJECTIVES = list(OBJECTIVE)
N_GROUPS = len(GROUP_LABELS)
table = runs(PACKAGE, tuple(OBJECTIVES))
assert len(table) == 30 and table.groupby(KEYS).seed.apply(set).eq({42, 43, 44}).all()

## 入力と学習ログの検算

run.json の best 選択と val AUROC 最大値、30 epoch の連続性、q の総和を確認する（`learning_tables`）。
予測 cache は checkpoint と split CSV の SHA256、画像の行順、target の一致を確かめてから読む（`prediction_tables`）。
global TPR/FPR は accuracy = p·TPR + (1−p)·TNR、2·BACC = TPR + TNR を解いて復元し、TP/FP が整数件数になることを検算する。
best epoch で、学習ログの値と cache から直接計算した値の差を記録する（学習時は bf16-mixed、cache は float32 なので完全一致はしない）。

In [ ]:
history, q_history, selections = learning_tables(table, PACKAGE)
history.to_csv(RESULTS / "validation_history_by_seed.csv", index=False)
q_history.to_csv(RESULTS / "q_history_by_seed.csv", index=False)
selections.to_csv(RESULTS / "best_epochs.csv", index=False)
test, groups, provenance = prediction_tables(table, PACKAGE, "test")
validation, val_groups, val_provenance = prediction_tables(table, PACKAGE, "val")

val_frame = load_split("val")
assert (val_frame.group >= 0).all(), "val に属性欠損は無いはず"
for metric, n in (("tpr", int(val_frame.target.sum())), ("fpr", int((val_frame.target == 0).sum()))):
    reconstructed_counts = history[metric].to_numpy() * n
    assert np.abs(reconstructed_counts - np.round(reconstructed_counts)).max() < 0.01

selected_history = history.merge(selections[["run_id", "best_epoch"]], on="run_id").query("epoch == best_epoch")
checks = selected_history.merge(validation, on="run_id", suffixes=("_log", "_cache"))
check_metrics = ["auroc", "bacc", "tpr", "fpr", "auroc_gap", "eopp1", "worst_group_auroc"]
errors = pd.DataFrame({"run_id": checks.run_id})
for metric in check_metrics:
    errors[metric] = checks[f"{metric}_log"] - checks[f"{metric}_cache"]
assert np.isfinite(errors[check_metrics].to_numpy()).all()
errors.to_csv(RESULTS / "validation_cache_log_errors.csv", index=False)

test.to_csv(RESULTS / "test_by_seed.csv", index=False)
groups.to_csv(RESULTS / "test_groups_by_seed.csv", index=False)
pd.concat([provenance, val_provenance]).to_csv(RESULTS / "provenance.csv", index=False)
counts = groups[["group", "group_label", "n_pos", "n_neg"]].drop_duplicates().sort_values("group")
assert len(counts) == N_GROUPS
counts.to_csv(RESULTS / "test_group_counts.csv", index=False)
print("test の評価行:", test.n_rows.unique(), "| 最大 cache/log 誤差:", np.abs(errors[check_metrics].to_numpy()).max())

## 集計と図の共通設定

gap や worst は各 seed 内で 4 群から求めてから平均する。群の seed 平均を先に取ってから gap を計算しない。
各図の点は seed、太いマーカーとエラーバーは平均 ± SD。群別図の陽性数は TPR、陰性数は FPR の分母。

In [ ]:
def aggregate(frame, keys, metrics):
    grouped = frame.groupby(keys, sort=False)[metrics]
    return grouped.mean().join(grouped.std().add_suffix("_sd")).reset_index()


means = aggregate(test, KEYS, FAIRNESS + GROUP_METRICS + ["eopp0", "worst_fpr", "auroc_all_rows"])
group_means = aggregate(groups, KEYS + ["group", "group_label", "n_pos", "n_neg"], GROUP_METRICS)
means.to_csv(RESULTS / "test_mean_sd.csv", index=False)
group_means.to_csv(RESULTS / "test_groups_mean_sd.csv", index=False)
history_means = aggregate(history, KEYS + ["epoch"], [m for m in FAIRNESS + GROUP_METRICS if m != "min_tpr"])
history_means.to_csv(RESULTS / "validation_history_mean_sd.csv", index=False)


def save_figure(fig, name):
    BOOK.savefig(fig)
    for extension in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{extension}")
    plt.show()
    plt.close(fig)


fair_limits = {}
for metric in FAIRNESS:
    lower = 0 if metric in LOWER_IS_BETTER else max(0, test[metric].min() - 0.05)
    upper = min(1, test[metric].max() + 0.03)
    fair_limits[metric] = (lower, upper)


def point_panel(ax, frame, metric, categories, category_column, labels):
    for x, category in enumerate(categories):
        values = frame.loc[frame[category_column] == category].sort_values("seed")[metric].to_numpy()
        assert len(values) == 3
        color = COLORS.get(category, f"C{x}")
        ax.scatter(x + np.linspace(-0.08, 0.08, 3), values, s=22, alpha=0.55, color=color)
        ax.errorbar(x, values.mean(), yerr=values.std(ddof=1), fmt="D", markersize=5, capsize=4, color=color)
    ax.set_xticks(range(len(categories)), [labels[c] for c in categories], fontsize=9)
    ax.set_title(TITLES[metric])
    ax.set_ylim(*fair_limits[metric])


def group_panels(frame, category_column, categories, labels, title, name):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
    for ax, metric in zip(axes.flat, GROUP_METRICS, strict=True):
        for i, category in enumerate(categories):
            selected = frame.loc[frame[category_column] == category]
            stats = selected.groupby("group")[metric].agg(["mean", "std"]).reindex(range(N_GROUPS))
            assert len(selected) == 3 * N_GROUPS and stats.notna().all().all()
            x = np.arange(N_GROUPS) + (i - (len(categories) - 1) / 2) * 0.18
            ax.errorbar(
                x,
                stats["mean"],
                yerr=stats["std"],
                fmt="o",
                capsize=3,
                label=labels[category],
                color=COLORS.get(category, f"C{i}"),
                markersize=5,
            )
        ax.set_title(TITLES[metric])
        ax.set_ylim(0, 1)
        tick_labels = [f"{r.group_label}\n+{r.n_pos} / −{r.n_neg}" for r in counts.itertuples()]
        ax.set_xticks(range(N_GROUPS), tick_labels)
    axes[0, 0].legend(loc="lower left", fontsize=9)
    fig.suptitle(
        title
        + "\nTest (attribute-complete) | best val AUROC checkpoint | mean ± SD, 3 seeds | + positives / − negatives",
        fontsize=13,
    )
    save_figure(fig, name)

## 1. ERM の公平性：モデル横断比較

1-stage と 2-stage を分ける。2-stage の ResNet は継続学習したモデルではなく、凍結前の親 ERM を参考として再掲する。
AUC gap・Eopp1 は低いほど、worst-AUC・worst-TPR は高いほどよい。

In [ ]:
erm = test.query("objective == 'erm'")
erm_groups = groups.query("objective == 'erm'")
erm_labels = {"resnet": "ResNet", "spatial_lora": "Spatial S3–4+fc", "spatial_lora_fc": "Spatial fc"}
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for row, procedure in enumerate(STAGE):
    selected = erm.loc[erm.procedure == procedure]
    if procedure == "two_stage":
        selected = pd.concat([selected, erm.query("model == 'resnet'")])
    for ax, metric in zip(axes[row], FAIRNESS, strict=True):
        point_panel(ax, selected, metric, MODELS, "model", erm_labels)
    axes[row, 0].set_ylabel(STAGE[procedure] + ("\nResNet = parent reference" if row else ""))
fig.suptitle(
    "1. ERM: does attribute conditioning improve fairness? (HAM10000, sex × age)"
    "\nTest (attribute-complete) | best val AUROC checkpoint | dots: seeds; diamonds: mean ± SD",
    fontsize=14,
)
save_figure(fig, "01_erm_fairness")

観察：1-stage の Spatial LoRA（S3–4＋fc）は ResNet より全体 AUROC（0.903→0.927）と worst-AUC（0.859→0.892）が高い。一方で Eopp1 は 0.196→0.259 と広がり、worst-TPR は 0.562→0.554 でほぼ同じ。fc のみは worst-TPR が 0.629 と高いが、seed 間 SD が ResNet 0.115・fc 0.103 あり、差は seed の揺れの内側にある。2-stage は両構成とも worst-TPR が 0.50 前後で、親 ResNet の平均より低い。

## 2. ERM の 4 群別性能

gap の縮小が最悪群の改善か、良い群の悪化かを切り分ける。群順と軸範囲は全図で固定する。

In [ ]:
for procedure in STAGE:
    selected = erm_groups.loc[erm_groups.procedure == procedure]
    if procedure == "two_stage":
        selected = pd.concat([selected, erm_groups.query("model == 'resnet'")])
    extra = " | ResNet = parent reference" if procedure == "two_stage" else ""
    group_panels(
        selected,
        "model",
        MODELS,
        MODEL,
        "2. ERM subgroup performance | " + STAGE[procedure] + extra,
        f"02_erm_groups_{procedure}",
    )

観察：TPR が最も低い群は、どの条件でも female × <65（test 陽性 35 件）。2-stage は全群で FPR が下がり（male × ≥65 で 0.239→0.169）、TPR も female × <65 で 0.562→0.495 に下がる。条件付けで動いたのは群間の差より、全体の動作点（見落としを増やして偽陽性を減らす側）に見える。

## 3. 公平性損失を加えた場合

同じ Spatial LoRA 構成・同じ学習手順の ERM を対照に、GroupDRO を並べる。
1-stage と 2-stage でそれぞれ比較するので、凍結の効果を損失の効果に混ぜない。

In [ ]:
for model in ("spatial_lora", "spatial_lora_fc"):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for row, procedure in enumerate(STAGE):
        selected = test.loc[(test.model == model) & (test.procedure == procedure)]
        for ax, metric in zip(axes[row], FAIRNESS, strict=True):
            point_panel(ax, selected, metric, OBJECTIVES, "objective", OBJECTIVE)
        axes[row, 0].set_ylabel(STAGE[procedure])
    fig.suptitle(
        "3. Fairness objectives | "
        + MODEL[model]
        + "\nTest (attribute-complete) | best val AUROC checkpoint | dots: seeds; diamonds: mean ± SD",
        fontsize=14,
    )
    save_figure(fig, f"03_objective_fairness_{model}")

観察：同じ Spatial LoRA・同じ stage の ERM に対し、GroupDRO は Eopp1 を下げ worst-TPR を上げる向きに動く（1-stage S3–4＋fc：Eopp1 −0.068、worst-TPR +0.043、2-stage S3–4＋fc：−0.048、+0.048、1-stage fc：−0.036、+0.033、同 seed 差の平均）。ただし同 seed 差の SD は 0.04〜0.12 で、平均と同じかそれより大きい。2-stage fc は ERM とほぼ同じ（差はほぼ 0）。

## 4. 公平性損失を加えた場合の 4 群別性能

TPR だけを上げ、同時に FPR も上げた場合は、その両方が分かるよう並べる。

In [ ]:
for model in ("spatial_lora", "spatial_lora_fc"):
    for procedure in STAGE:
        selected = groups.loc[(groups.model == model) & (groups.procedure == procedure)]
        group_panels(
            selected,
            "objective",
            OBJECTIVES,
            OBJECTIVE,
            "4. Objectives by subgroup | " + MODEL[model] + " | " + STAGE[procedure],
            f"04_objective_groups_{model}_{procedure}",
        )

観察：1-stage S3–4＋fc では、GroupDRO で female × <65 の TPR が 0.562→0.600 に上がるのと同時に、female × ≥65 の FPR が 0.084→0.156 に上がる。gap の縮小は最悪群の TPR の上昇によるものだが、偽陽性の増加を伴う。

## 5. 1 run の GroupDRO 重み推移

CheXpert と同じく、Spatial LoRA (stage3–4 + fc)、1-stage、seed 42 を事前に指定する。
左は epoch 内の平均 q、右は epoch 末の q。q は確率分布なので各時点で 4 群の和が 1。
縦破線は best val AUROC の epoch で、test の結果を見て選んだ時点ではない。

In [ ]:
chosen = table.query(
    "model == 'spatial_lora' and procedure == 'one_stage' and objective == 'group_dro' and seed == 42"
).iloc[0]
chosen_q = q_history.loc[q_history.run_id == chosen.run_id]
best_epoch = selections.loc[selections.run_id == chosen.run_id, "best_epoch"].item()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, kind, label in zip(axes, ("q_mean", "q_end"), ("Mean within epoch", "End of epoch"), strict=True):
    for g, group_label in enumerate(GROUP_LABELS):
        line = chosen_q.loc[(chosen_q.kind == kind) & (chosen_q.group == g)]
        ax.plot(line.epoch, line.q, label=group_label, linestyle="-" if g < 2 else "--")
    ax.axhline(1 / N_GROUPS, color="0.6", linestyle=":", linewidth=1)
    ax.axvline(best_epoch, color="0.3", linestyle="--", linewidth=1, label=f"Best epoch = {best_epoch}")
    ax.set(xlabel="Epoch (1-based)", ylabel="Group weight q", title=label, ylim=(0, 1), xlim=(1, 30))
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
fig.suptitle("5. GroupDRO weights | Spatial LoRA (S3–4 + fc), 1-stage, seed 42", fontsize=13)
save_figure(fig, "05_group_dro_weights")
chosen_q.to_csv(RESULTS / "selected_q_history.csv", index=False)

観察：q はエポックを追うごとに male × ≥65 へ寄り（epoch 末で 0.263→0.342）、test で TPR が最も低い female × <65 は一様（0.25）より低い 0.21 前後にとどまる。q は train の損失に対する重みで、train の損失が大きい群と、固定しきい値で見落としが多い群は一致していない。

## 6. validation の学習推移

ERM / GroupDRO を同一モデル・同一 stage で比較する。
横軸は 1 始まりの epoch（2-stage では補正学習開始からの epoch）。線は seed 平均、帯は ± SD。
上段が公平性、下段が global 指標。

**worst-TPR の全 epoch 値は未記録**（ログは群の最小 TPR を持たず、ERM/GroupDRO は途中の checkpoint も残っていない）。
その枠には各 seed の best checkpoint で直接計算した点だけを表示し、線で結ばない。この点は seed 平均ではない。

In [ ]:
for procedure, model in table[["procedure", "model"]].drop_duplicates().itertuples(index=False, name=None):
    selected = history_means.loc[(history_means.procedure == procedure) & (history_means.model == model)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, metric in zip(axes.flat, FAIRNESS + GROUP_METRICS, strict=True):
        ax.set_title(("Global " if metric in GROUP_METRICS else "") + TITLES[metric])
        for objective in OBJECTIVES:
            color = COLORS[objective]
            if metric == "min_tpr":
                dots = validation.loc[
                    (validation.procedure == procedure)
                    & (validation.model == model)
                    & (validation.objective == objective)
                ].merge(selections[["run_id", "best_epoch"]], on="run_id")
                ax.scatter(dots.best_epoch, dots.min_tpr, color=color, s=30, label=OBJECTIVE[objective])
                continue
            line = selected.loc[selected.objective == objective].sort_values("epoch")
            x, y, sd = (line[c].to_numpy() for c in ("epoch", metric, metric + "_sd"))
            ax.plot(x, y, color=color, label=OBJECTIVE[objective])
            ax.fill_between(x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=color, alpha=0.15)
        ax.set(xlabel="Epoch (1-based)", xlim=(0.5, 30.5))
        if metric in LOWER_IS_BETTER:
            ax.set_ylim(bottom=0)
        if metric == "min_tpr":
            ax.set_title("Worst-TPR ↑ | best-epoch points ONLY", fontsize=10)
            ax.text(0.96, 0.12, "Full history not recorded", transform=ax.transAxes, ha="right", fontsize=9)
    axes[0, 0].legend(fontsize=9)
    fig.suptitle(
        "6. Validation learning curves | "
        + MODEL[model]
        + " | "
        + STAGE[procedure]
        + "\nLines: 3-seed mean ± SD; worst-TPR: individual best-checkpoint values",
        fontsize=13,
    )
    save_figure(fig, f"06_learning_{model}_{procedure}")

観察：1-stage では validation の Eopp1 が epoch ごとに 0.2〜0.35 の幅で揺れ、ERM と GroupDRO の曲線は帯が重なる。test で見た 0.05 前後の差は、この epoch 間の揺れより小さい。全体 TPR は学習とともに下がり、FPR も下がる（動作点が特異度の側へ動く）。2-stage の best epoch は大半が 1〜4 で、補正の学習が始まってすぐに選ばれている。

## 対応する seed の差と観察用の表

モデル差は同 seed の親 ResNet ERM、損失差は同モデル・同 stage の ERM を基準にする。
平均 ± SD だけで「効果なし」「有意に改善」と断定しない。全体性能と TPR/FPR の変化も併記する。

In [ ]:
metrics = FAIRNESS + GROUP_METRICS + ["eopp0", "worst_fpr"]
parent = test.query("model == 'resnet' and objective == 'erm'").set_index("seed")
paired = []
for procedure, model, objective in test[KEYS].drop_duplicates().itertuples(index=False, name=None):
    selected = test.loc[
        (test.procedure == procedure) & (test.model == model) & (test.objective == objective)
    ].set_index("seed")
    if objective == "erm":
        reference, baseline = parent, "parent ResNet ERM"
    else:
        reference = test.loc[
            (test.procedure == procedure) & (test.model == model) & (test.objective == "erm")
        ].set_index("seed")
        baseline = "same-model same-stage ERM"
    delta = selected[metrics] - reference[metrics]
    for seed, row in delta.iterrows():
        paired.append(
            {
                "procedure": procedure,
                "model": model,
                "objective": objective,
                "baseline": baseline,
                "seed": seed,
                **row.to_dict(),
            }
        )
paired = pd.DataFrame(paired)
paired.to_csv(RESULTS / "paired_deltas_by_seed.csv", index=False)
delta_means = aggregate(paired, KEYS + ["baseline"], metrics)
delta_means.to_csv(RESULTS / "paired_deltas_mean_sd.csv", index=False)

## 図一覧の書き出し

PNG を 1 つの HTML に埋め込み、PDF をまとめて閉じる。

In [ ]:
BOOK.close()
images = []
for path in sorted(FIGURES.glob("*.png")):
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    images.append(
        f'<section id="{path.stem}"><h2>{path.stem}</h2>'
        f'<img src="data:image/png;base64,{encoded}" alt="{path.stem}"></section>'
    )
html = """<!doctype html><html lang="ja"><meta charset="utf-8"><title>HAM10000 公平性の比較</title>
<style>body{max-width:1500px;margin:32px auto;padding:0 24px;font-family:system-ui;background:#fafafa;color:#18212b}
img{width:100%;height:auto}
section{background:white;padding:20px;margin:32px 0;border:1px solid #ddd}
h2{font-size:18px}p{line-height:1.8}</style>
<h1>属性条件付き予測と公平性（HAM10000、ERM・GroupDRO）</h1>
<p>HAM10000・sex × 年齢 65 歳の 4 群・3 seed。
図1–4は best validation AUROC checkpoint の test 評価（属性の揃った 1,217 件）、
図6は validation の学習推移。エラーバーと帯は seed 間 SD。陽性は悪性、worst-TPR は 4 群の最小 TPR。<br>
図6の worst-TPR は best epoch の点のみ（全 epoch の値は未記録）。EO rate は回し直し中で含めていない。
図2・4の群名には陽性・陰性件数を併記。</p>"""
(FIGURES / "index.html").write_text(html + "".join(images) + "</html>", encoding="utf-8")
print("Saved PNG/PDF figures:", len(images))